# MvDEC 2025 REUTERS ambiguity matrix — seed 42

Runs B1-B3 first, then the two full-batch isolation protocols F0/P0. B0 is not rerun because its verified artifact already exists. Full-batch failures are reported rather than silently replaced by gradient accumulation.


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPUs:", gpus)
assert gpus, "Enable a Kaggle/Colab GPU before running this notebook."


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

TRAINING_COMMIT = "9b63446a0dd535a8ab94afc1e5457ab1594fb9ef"
REPO = Path("/kaggle/working/mvdec_fpmax")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "https://github.com/vdat-18/mvdec_fpmax.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", "--detach", TRAINING_COMMIT], cwd=REPO, check=True)
resolved = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert resolved == TRAINING_COMMIT, (resolved, TRAINING_COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "gdown"], check=True)
subprocess.run([sys.executable, str(REPO / "scripts/fetch_dekm_release.py"), "--datasets", "REUTERS"], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_mvdec_contract.py", "tests/test_mvdec_preprocessing.py", "tests/test_mvdec_architecture.py", "-k", "equation11 or direct_joint_head or paper_informed"], cwd=REPO, check=True)


In [ ]:
OUTPUT = Path("/kaggle/working/mvdec-ambiguity-seed42")
COMMON = [
    sys.executable, "-u", "src/representation_learning/MVDEC_dense.py", "REUTERS",
    "--runs", "1", "--seed", "42", "--lambda-kmeans", "1",
    "--lambda-greedy", "1", "--progress-interval", "100",
    "--dataset-root", "external_repos/DEKM/datasets",
    "--public-output-dir", str(OUTPUT),
]
PROTOCOLS = {
    "B1": ("mvdec_2025_equation11_smallest_frozen_v1", "smallest", "frozen_snapshot"),
    "B2": ("mvdec_2025_equation11_largest_selected_only_v1", "largest", "selected_dimension_only"),
    "B3": ("mvdec_2025_equation11_smallest_selected_only_v1", "smallest", "selected_dimension_only"),
    "F0": ("mvdec_2025_equation11_largest_frozen_full_batch_v1", "largest", "frozen_snapshot"),
    "P0": ("mvdec_2025_paper_resolved_full_batch_v1", "smallest", "selected_dimension_only"),
}

def run_protocol(label):
    protocol_id, eigen, target = PROTOCOLS[label]
    command = COMMON + [
        "--protocol", protocol_id,
        "--greedy-eigen-direction", eigen,
        "--greedy-target-mode", target,
    ]
    print("RUN", label, protocol_id, flush=True)
    completed = subprocess.run(command, cwd=REPO)
    print("RETURN CODE", label, completed.returncode, flush=True)
    return completed.returncode


## B1-B3: eigen/target ambiguity

These runs retain batch size 256 and isolate only eigen direction and greedy-target semantics.


In [ ]:
return_codes = {label: run_protocol(label) for label in ("B1", "B2", "B3")}
assert all(code == 0 for code in return_codes.values()), return_codes


## F0: full-batch isolation

F0 differs from B0 only in refinement batch policy. If a 16 GB T4 cannot allocate the full gradient batch, keep the failure as evidence.


In [ ]:
return_codes["F0"] = run_protocol("F0")


## P0: paper-resolved full batch

P0 combines the explicit smallest-eigenvalue statement, selected-dimension-only L4, and the Table 2 `DEKM_F` full-batch clue.


In [ ]:
return_codes["P0"] = run_protocol("P0")


In [ ]:
import json
import pickle
import re
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

rows = []
for label, (protocol_id, eigen, target) in PROTOCOLS.items():
    roots = sorted((OUTPUT / "reuters" / protocol_id).glob("*/seed_42"))
    if not roots:
        rows.append({"label": label, "protocol_id": protocol_id, "status": "no_artifact", "return_code": return_codes.get(label)})
        continue
    run_dir = roots[-1]
    manifest = json.loads((run_dir / "manifest.json").read_text())
    assert manifest["training_commit"] == TRAINING_COMMIT
    row = {
        "label": label,
        "protocol_id": protocol_id,
        "status": manifest["status"],
        "return_code": return_codes.get(label),
        "runtime_seconds": manifest.get("runtime_seconds"),
        "config_hash": manifest.get("config_hash"),
        "training_commit": manifest.get("training_commit"),
        "source_id": manifest.get("source_id"),
        **manifest.get("metrics", {}),
    }
    if manifest["status"] == "complete":
        with (run_dir / manifest["paths"]["artifact"]).open("rb") as handle:
            artifact = pickle.load(handle)
        labels = np.asarray(artifact["labels"])
        fused = np.asarray(artifact["h_fused"])
        row.update({
            "resolved_batch_size": artifact.get("resolved_refinement_batch_size", artifact["config"]["batch_size"]),
            "batches_per_epoch": artifact["batches_per_epoch"],
            "stop_reason": artifact["stop_reason"],
            "training_steps": artifact.get("training_steps_completed"),
            "cluster_sizes": np.bincount(labels, minlength=artifact["n_clusters"]).tolist(),
            "view1_mean_norm": float(np.linalg.norm(artifact["h_view1"], axis=1).mean()),
            "view2_mean_norm": float(np.linalg.norm(artifact["h_view2"], axis=1).mean()),
            "fused_mean_norm": float(np.linalg.norm(fused, axis=1).mean()),
            "view1_total_variance": float(np.var(artifact["h_view1"], axis=0).sum()),
            "view2_total_variance": float(np.var(artifact["h_view2"], axis=0).sum()),
            "fused_total_variance": float(np.var(fused, axis=0).sum()),
            "fused_inertia": float(KMeans(n_clusters=artifact["n_clusters"], n_init=100, random_state=42).fit(fused).inertia_),
            "fused_silhouette": float(silhouette_score(fused, labels)),
        })
        log_text = (run_dir / manifest["paths"]["training_log"]).read_text(errors="replace")
        loss_lines = [line for line in log_text.splitlines() if "L1_reconstruction:" in line]
        row["final_loss_log"] = loss_lines[-1] if loss_lines else None
    rows.append(row)

summary = pd.DataFrame(rows)
pd.set_option("display.max_columns", None)
display(summary)
summary.to_json(OUTPUT / "ambiguity_matrix_summary.json", orient="records", indent=2)
summary.to_csv(OUTPUT / "ambiguity_matrix_summary.csv", index=False)


In [ ]:
import hashlib
import shutil

zip_base = "/kaggle/working/mvdec-ambiguity-reuters-seed42"
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT.parent, OUTPUT.name)
digest = hashlib.sha256(Path(zip_path).read_bytes()).hexdigest()
print("ZIP:", zip_path)
print("SHA-256:", digest)
print("Return codes:", return_codes)
